# Eje 4a — Market Data US: spot, quote y cadena

Derivados Financieros — QUANt UCEMA

Mismos módulos que la página **Market Data → NYSE** de UCEMA QUANT: `Codigo.data.market_data`.

| Notebook | Contenido |
|----------|-----------|
| **04a** (este) | Spot / quote / cadena **NYSE–US** |
| **04b** | Lectura del panel: **IV ATM** y descriptivos |
| **04c** | **BYMA**: spot ARS, paneles y opciones (`byma_market`) |

**Plan del notebook:**
1. Por qué un módulo unificado
2. Spot y quote
3. Expiries y options chain
4. Mid, spread y ATM
5. Resumen


## 1) Por qué un módulo unificado

`get_spot` / `get_quote` / `get_expirations` / `get_options_chain` prueban varias fuentes y degradan con gracia. En clase: **siempre** try/except + mensaje.


In [1]:
import sys
sys.path.append('../../..')

import pandas as pd
from Codigo.data.market_data import (
    get_spot, get_quote, get_expirations, get_options_chain,
)

TICKER = 'MSFT'


## 2) Spot y quote


In [2]:
try:
    spot = get_spot(TICKER)
    q = get_quote(TICKER)
    print(f'{TICKER} spot={spot:.2f}')
    if isinstance(q, dict):
        print({k: q.get(k) for k in ('price', 'change', 'change_pct', 'source')})
except Exception as e:
    spot, q = None, None
    print('Fallo spot/quote:', type(e).__name__, e)


MSFT spot=493.78
{'price': None, 'change': -3.9700012, 'change_pct': None, 'source': 'yahooquery'}


## 3) Expiries y options chain


In [3]:
try:
    exps = get_expirations(TICKER)
    print(f'{len(exps)} expiries')
    expiry = exps[0]
    chain = get_options_chain(TICKER, expiry)
    print(f'chain {expiry}: shape={chain.shape}')
    print(list(chain.columns))
    display(chain.head())
except Exception as e:
    exps, expiry, chain = [], None, None
    print('Fallo cadena:', type(e).__name__, e)


22 expiries


chain 2026-09-21: shape=(79, 19)
['symbol', 'expiration', 'type', 'contractSymbol', 'strike', 'currency', 'lastPrice', 'change', 'percentChange', 'volume', 'openInterest', 'bid', 'ask', 'contractSize', 'lastTradeDate', 'impliedVolatility', 'inTheMoney', 'Spot', 'Ticker']


,symbol,expiration,type,contractSymbol,strike,currency,lastPrice,change,percentChange,volume,openInterest,bid,ask,contractSize,lastTradeDate,impliedVolatility,inTheMoney,Spot,Ticker
0,MSFT,2026-09-21,call,MSFT260921C00430000,430.0,USD,64.60,2.299999,3.691813,2.0,5.0,62.40,65.30,REGULAR,2026-09-18 19:42:05,0.775393,True,493.779999,MSFT
1,MSFT,2026-09-21,call,MSFT260921C00435000,435.0,USD,61.54,1.790001,2.995817,3.0,6.0,57.40,60.15,REGULAR,2026-09-18 18:52:33,1.189457,True,493.779999,MSFT
2,MSFT,2026-09-21,call,MSFT260921C00440000,440.0,USD,54.65,-6.500000,-10.629599,4.0,1.0,52.55,55.10,REGULAR,2026-09-18 18:11:14,0.625004,True,493.779999,MSFT
3,MSFT,2026-09-21,call,MSFT260921C00450000,450.0,USD,43.63,-0.040001,-0.091598,16.0,0.0,42.65,44.70,REGULAR,2026-09-18 19:58:57,0.850099,True,493.779999,MSFT
4,MSFT,2026-09-21,call,MSFT260921C00455000,455.0,USD,41.28,0.000000,0.000000,0.0,1.0,37.45,40.30,REGULAR,2026-09-11 17:27:13,0.508794,True,493.779999,MSFT


## 4) Mid, spread y ATM

Para pricing de estrategias usamos mid (o last). El ATM es el strike más cercano al spot.


In [4]:
if chain is not None and spot is not None and len(chain):
    df = chain.copy()
    if {'call_bid', 'call_ask'}.issubset(df.columns):
        df['call_mid'] = (df['call_bid'] + df['call_ask']) / 2
        df['call_spread'] = df['call_ask'] - df['call_bid']
    if {'put_bid', 'put_ask'}.issubset(df.columns):
        df['put_mid'] = (df['put_bid'] + df['put_ask']) / 2
        df['put_spread'] = df['put_ask'] - df['put_bid']
    atm_i = (df['strike'] - spot).abs().idxmin()
    print(f'ATM strike ≈ {df.loc[atm_i, "strike"]}')
    cols = [c for c in ['strike', 'call_mid', 'call_spread', 'put_mid', 'put_spread',
                        'call_iv', 'put_iv'] if c in df.columns]
    display(df.loc[[atm_i], cols] if cols else df.loc[[atm_i]])
else:
    print('Sin datos — abrí la página Market Data de la app como alternativa.')


ATM strike ≈ 495.0


,strike
20,495.0


## Resumen

- Flujo app = flujo notebook: spot → expiries → chain → mid/ATM (**mercado US**).
- **Ejercicio:** ¿qué spread bid-ask te parece “caro” para armar un butterfly?
- **Siguiente:** **04b** — IV ATM y lectura del panel · **04c** — paneles BYMA / data912. Página app: *Market Data*.
